### **Import necessary libraries**

In [1]:
import os 
import sys 

import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
%matplotlib inline

In [2]:
current_cwd = os.getcwd()
current_cwd
new_cwd = '\\'.join(current_cwd.split('\\')[:2])
sys.path.append(new_cwd + '\\utils')

In [3]:
from helper import *

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### **Load the dataset**

In [4]:
from datasets import load_dataset

dataset = load_dataset("ncduy/mt-en-vi")

C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\datasets--ncduy--mt-en-vi. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 2884451
    })
    validation: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 11316
    })
    test: Dataset({
        features: ['en', 'vi', 'source'],
        num_rows: 11225
    })
})

In [6]:
train = dataset['train'].to_pandas()
val = dataset['validation'].to_pandas()
test = dataset['test'].to_pandas()

In [7]:
envi_data = pd.concat(
    [
        train, 
        val,
        test
    ]
)
envi_data.reset_index(drop=True, inplace=True)

In [8]:
envi_data.head()

,en,vi,source
0,"- Sorry, that question's not on here.","- Xin lỗi, nhưng mà ở đây không có câu hỏi đấy.",OpenSubtitles v2018
1,He wants you to come with him immediately.,Ông ấy muốn bố đi với ông ấy ngay lập tức,OpenSubtitles v2018
2,I thought we could use some company.,Tôi nghĩ chúng ta có thể muốn vài người bạn đồ...,OpenSubtitles v2018
3,It was founded in 2008 by this anonymous progr...,Nó được sáng lập vào năm 2008 bởi một lập trìn...,TED2020 v1
4,"With both of these methods, no two prints are ...","Với cả hai phương pháp, không có hai bản in nà...",TED2020 v1


In [9]:
envi_data.tail()

,en,vi,source
2906987,It's important.,Đây là điều quan trọng.,TED2020 v1
2906988,I looked at him and I saw myself.,"Mình nhìn vào nó, và nhìn thấy chính con người...",OpenSubtitles v2018
2906989,In India it is distributed mainly on the plain...,"Tại Ấn Độ, nó phân bố chủ yếu trên các vùng đồ...",WikiMatrix v1
2906990,He moved to MIO Biwako Shiga in 2015.,Anh chuyển đến MIO Biwako Shiga năm 2015.,WikiMatrix v1
2906991,The words genius and musical are used in the s...,Những từ ngữ như thiên tài âm nhạc đã được sử ...,wikimedia v20210402


In [10]:
envi_data.shape

(2906992, 3)

### **Check for duplicated values**

In [12]:
envi_data.duplicated().sum()

np.int64(0)

### **Check for missing values**

In [13]:
envi_data.isnull().sum()

en        0
vi        0
source    0
dtype: int64

### **Filter dataset**

We want to filter the dataset to include only rows where both the English and Vietnamese sentences have a maximum length of 50 words, normalize sentence and contain no special characters.

In [17]:
# Normalize the sentences in both columns
envi_data["en"] = envi_data["en"].apply(normalize_sentence)
envi_data["vi"] = envi_data["vi"].apply(normalize_sentence)

# Filter the DataFrame: English sentences should have at most 50 words and be all ASCII,
# while Vietnamese sentences should have at most 50 words and only allowed characters.
filtered_envi_data = envi_data[
    (envi_data["en"].apply(lambda s: len(s.split()) <= 50 and s.isascii()))
    & (envi_data["vi"].apply(lambda s: len(s.split()) <= 50 and is_valid_vietnamese(s)))
]

# Check the shape of the filtered dataset
print(filtered_envi_data.shape)


(2635966, 3)


In [18]:
filtered_envi_data.head(10)

,en,vi,source
0,"- sorry, that question's not on here.","- xin lỗi, nhưng mà ở đây không có câu hỏi đấy.",OpenSubtitles v2018
1,he wants you to come with him immediately.,ông ấy muốn bố đi với ông ấy ngay lập tức,OpenSubtitles v2018
2,i thought we could use some company.,tôi nghĩ chúng ta có thể muốn vài người bạn đồ...,OpenSubtitles v2018
3,it was founded in 2008 by this anonymous progr...,nó được sáng lập vào năm 2008 bởi một lập trìn...,TED2020 v1
4,"with both of these methods, no two prints are ...","với cả hai phương pháp, không có hai bản in nà...",TED2020 v1
5,from these contexts was born an installation i...,từ những tình huống này một bố trí không gian ...,TED2020 v1
6,i have lived to see something which i never ex...,ta đã sống để thấy điều ta không bao giờ mong ...,OpenSubtitles v2018
7,it is the model for all future relationships w...,đó là mô hình cho tất cả các mối quan hệ trong...,TED2020 v1
8,welcome him as your brother.,chào mừng nó như anh em của các con.,OpenSubtitles v2018
9,so biologists can make all the mutant fruit fl...,vậy các nhà sinh vật học có thể biến đổi gene ...,TED2020 v1


### **Randomly select 300k (or 500k) rows from the filtered dataset**

In [ ]:
import pandas as pd

df = filtered_envi_data.sample(n=300000, random_state=42)
# df = filtered_envi_data.sample(n=500000, random_state=42)

df.to_csv('dataset_300k.csv', index=False)
# df.to_csv('dataset_500k.csv', index=False)